# Industrial PCB Anomaly Detection — PatchCore Research Pipeline

An optional, reproducible anomaly-detection experiment using VisA `pcb1`,
`pcb2`, and `pcb3`.

**Pipeline:** local dataset → ImageNet ResNet50 layer2/layer3 patches →
per-category PCA → PatchCore coreset + KNN baseline → industrial metrics →
research artifacts.

Every data, cache, artifact, and result path is relative to the Git
repository. The runnable reference-comparison app is independent of these
research artifacts.


## Environment setup

Start Jupyter from the cloned repository root. Use a CUDA GPU when
available; full feature extraction on CPU is supported but can take hours.


In [ ]:
# Run once per Python environment, then restart the kernel if requested.
%pip install -q -r requirements-train.txt


In [ ]:
from __future__ import annotations

import gc
import json
import logging
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

try:
    import faiss
except ImportError:
    faiss = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

current_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)
DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = PROJECT_ROOT / ".cache" / "raw"
FEATURE_ROOT = PROJECT_ROOT / ".cache" / "features"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
RESULT_ROOT = PROJECT_ROOT / "results"

for folder in (
    DATA_ROOT,
    RAW_ROOT,
    FEATURE_ROOT,
    ARTIFACT_ROOT,
    RESULT_ROOT,
):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CATEGORIES = ["pcb1", "pcb2", "pcb3"]

IMG_SIZE = 256
BATCH_SIZE = 8
PATCH_STRIDE = 2
PCA_VARIANCE = 0.95
PCA_FIT_PATCHES = 30_000
CORESET_RATIO = 0.01
CORESET_MAX = 5_000
KNN_MEMORY_MAX = 50_000
CALIBRATION_FRACTION = 0.10
CALIBRATION_QUANTILE = 0.90  # recall-oriented industrial operating point
GAUSSIAN_SIGMA = 4.0

# None means the complete split. Use small integers only for smoke tests.
DEBUG_MAX_TRAIN_IMAGES = None
DEBUG_MAX_TEST_IMAGES = None
AUGMENTATION_ENABLED = False
AUGMENTED_COPIES = 3

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)


In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU mode: correct but slower for feature extraction.")


## Phase 1 — Download, adapt, and persist the datasets


In [ ]:
import logging

from datasets import load_dataset
from huggingface_hub import login

# Optional authentication for higher Hub rate limits. Set the HF_TOKEN
# environment variable if desired. VisA is public and needs no login.
hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face Hub authentication enabled.")
else:
    class _HideUnauthenticatedHubNotice(logging.Filter):
        def filter(self, record):
            return "unauthenticated requests" not in record.getMessage().lower()

    logging.getLogger("huggingface_hub.utils._http").addFilter(
        _HideUnauthenticatedHubNotice()
    )
    print("Using public VisA access without a token.")


def _is_valid_image(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except (OSError, SyntaxError, ValueError):
        return False


def _atomic_png_save(image, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    image.save(temporary, format="PNG")
    if not _is_valid_image(temporary):
        temporary.unlink(missing_ok=True)
        raise OSError(f"Failed integrity check while writing {destination}")
    temporary.replace(destination)


def _save_rgb(image, destination: Path, force=False):
    if force or not _is_valid_image(destination):
        converted = image.convert("RGB")
        converted.load()
        _atomic_png_save(converted, destination)


def _save_binary_mask(mask, destination: Path, force=False):
    if force or not _is_valid_image(destination):
        array = np.asarray(mask.convert("L"))
        _atomic_png_save(
            Image.fromarray(np.uint8(array > 0) * 255),
            destination,
        )


def download_and_adapt_visa(destination: Path, force=False):
    # BrachioLab/visa packages the official VisA 1-class images and masks as
    # category-specific Parquet splits. Only pcb1, pcb2, and pcb3 are requested.
    for category in CATEGORIES:
        for split in ("train", "test"):
            dataset = load_dataset(
                "BrachioLab/visa",
                split=f"{category}.{split}",
                cache_dir=str(RAW_ROOT / "visa_hf_cache"),
            )
            for index, row in enumerate(
                tqdm(dataset, desc=f"VisA {category}.{split}")
            ):
                label = int(row["label"])
                defect = "good" if label == 0 else "bad"
                stem = f"{split}_{index:05d}"
                image_path = (
                    destination / category / split / defect / f"{stem}.png"
                )
                _save_rgb(row["image"], image_path, force=force)

                if split == "test" and label == 1 and row.get("mask") is not None:
                    mask_path = (
                        destination / category / "ground_truth" / "bad"
                        / f"{stem}_mask.png"
                    )
                    _save_binary_mask(row["mask"], mask_path, force=force)


def validate_pcb_dataset(root: Path):
    rows = []
    for category in CATEGORIES:
        train_good = sorted((root / category / "train" / "good").glob("*.png"))
        test_good = sorted((root / category / "test" / "good").glob("*.png"))
        test_bad = sorted((root / category / "test" / "bad").glob("*.png"))
        masks = sorted(
            (root / category / "ground_truth" / "bad").glob("*.png")
        )
        assert train_good, f"{category}: no nominal training images"
        assert test_good and test_bad, f"{category}: incomplete test split"
        assert len(masks) == len(test_bad), (
            f"{category}: {len(test_bad)} anomalies but {len(masks)} masks"
        )
        files = train_good + test_good + test_bad + masks
        corrupted = [path for path in files if not _is_valid_image(path)]
        assert not corrupted, (
            f"{category}: unreadable files remain: "
            + ", ".join(map(str, corrupted[:10]))
        )
        rows.append(
            {
                "category": category,
                "train_good": len(train_good),
                "test_good": len(test_good),
                "test_defective": len(test_bad),
                "masks": len(masks),
            }
        )
    return pd.DataFrame(rows)


def prepare_datasets(force=False):
    download_and_adapt_visa(DATA_ROOT, force=force)
    return validate_pcb_dataset(DATA_ROOT)


In [ ]:
# Downloads are idempotent. Existing valid local files are reused, while
# incomplete PNGs are regenerated atomically from cached VisA examples.
dataset_inventory = prepare_datasets(force=False)
display(dataset_inventory)


## Phase 2 — Multi-scale ResNet50 patch feature extraction and caching


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
CLASSIC_AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=12, scale=(0.92, 1.08)),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5)),
])


class InspectionDataset(Dataset):
    def __init__(self, paths, augment=False, augmented_copies=0):
        self.paths = list(paths)
        self.augment = augment
        self.multiplier = 1 + (augmented_copies if augment else 0)

    def __len__(self):
        return len(self.paths) * self.multiplier

    def __getitem__(self, index):
        base_index, variant = divmod(index, self.multiplier)
        path = self.paths[base_index]
        try:
            with Image.open(path) as source:
                source.load()
                image = source.convert("RGB")
        except (OSError, SyntaxError, ValueError) as error:
            raise OSError(
                f"Unreadable image: {path}. Rerun the Phase 1 preparation "
                "cell to validate and repair the persistent dataset."
            ) from error
        if variant > 0:
            image = CLASSIC_AUGMENT(image)
        key = str(path) if variant == 0 else f"{path}::aug{variant}"
        return BASE_TRANSFORM(image), key


class ResNetPatchExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.stem = torch.nn.Sequential(
            net.conv1, net.bn1, net.relu, net.maxpool, net.layer1
        )
        self.layer2, self.layer3 = net.layer2, net.layer3
        self.eval()
        for parameter in self.parameters():
            parameter.requires_grad_(False)

    @torch.inference_mode()
    def forward(self, batch):
        x = self.stem(batch)
        raw_f2 = self.layer2(x)
        raw_f3 = self.layer3(raw_f2)
        f2 = F.avg_pool2d(raw_f2, 3, 1, 1)
        f3 = F.avg_pool2d(raw_f3, 3, 1, 1)
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
        patches = torch.cat([f2, f3], dim=1)
        patches = patches[:, :, ::PATCH_STRIDE, ::PATCH_STRIDE]
        return patches.permute(0, 2, 3, 1).contiguous()


EXTRACTOR = ResNetPatchExtractor().to(DEVICE)


def image_paths(category, split):
    paths = sorted((DATA_ROOT / category / split).glob("*/*.png"))
    limit = DEBUG_MAX_TRAIN_IMAGES if split == "train" else DEBUG_MAX_TEST_IMAGES
    return paths[:limit] if limit else paths


def cache_dir(category, split, augment):
    tag = f"{split}_aug{AUGMENTED_COPIES}" if augment else split
    return FEATURE_ROOT / category / tag


@torch.inference_mode()
def extract_and_cache(category, split, augment=False, rebuild=False):
    output_dir = cache_dir(category, split, augment)
    manifest_path = output_dir / "manifest.json"
    if manifest_path.exists() and not rebuild:
        return json.loads(manifest_path.read_text())
    if rebuild and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = image_paths(category, split)
    dataset = InspectionDataset(
        paths, augment=augment and split == "train",
        augmented_copies=AUGMENTED_COPIES,
    )
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    chunks, grid = [], None
    for chunk_index, (batch, keys) in enumerate(tqdm(loader, desc=f"{category}:{split}")):
        feature_grid = EXTRACTOR(batch.to(DEVICE, non_blocking=True)).cpu().numpy()
        grid = list(feature_grid.shape[1:3])
        filename = f"chunk_{chunk_index:04d}.npz"
        np.savez_compressed(
            output_dir / filename,
            features=feature_grid.astype(np.float16),
            paths=np.asarray(list(keys)),
        )
        chunks.append(filename)
    manifest = {
        "category": category, "split": split, "augment": bool(augment),
        "chunks": chunks, "grid": grid, "channels": 1536,
        "images": len(dataset), "img_size": IMG_SIZE,
        "patch_stride": PATCH_STRIDE,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return manifest


def iter_cached(category, split, augment=False):
    directory = cache_dir(category, split, augment)
    manifest = json.loads((directory / "manifest.json").read_text())
    for filename in manifest["chunks"]:
        with np.load(directory / filename) as chunk:
            yield chunk["features"].astype(np.float32), chunk["paths"].astype(str)


In [ ]:
feature_manifests = {}
for category in CATEGORIES:
    feature_manifests[(category, "train")] = extract_and_cache(
        category, "train", augment=AUGMENTATION_ENABLED
    )
    feature_manifests[(category, "test")] = extract_and_cache(
        category, "test", augment=False
    )
pd.DataFrame(feature_manifests.values())


## Phase 3 — Per-category PCA retaining 95% variance


In [ ]:
def sample_training_patches(category, augment, maximum=PCA_FIT_PATCHES):
    chunks = list(iter_cached(category, "train", augment))
    patches_per_chunk = max(1, math.ceil(maximum / max(len(chunks), 1)))
    rng = np.random.default_rng(SEED)
    sampled = []
    for features, _ in chunks:
        flat = features.reshape(-1, features.shape[-1])
        count = min(patches_per_chunk, len(flat))
        sampled.append(flat[rng.choice(len(flat), count, replace=False)])
    sampled = np.concatenate(sampled)
    if len(sampled) > maximum:
        sampled = sampled[rng.choice(len(sampled), maximum, replace=False)]
    return sampled.astype(np.float32)


def fit_category_pca(category, augment=False, rebuild=False):
    category_dir = ARTIFACT_ROOT / category
    category_dir.mkdir(parents=True, exist_ok=True)
    pca_path = category_dir / "pca.joblib"
    if pca_path.exists() and not rebuild:
        return joblib.load(pca_path)
    sampled = sample_training_patches(category, augment)
    pca = PCA(n_components=PCA_VARIANCE, svd_solver="full")
    pca.fit(sampled)
    joblib.dump(pca, pca_path)
    print(
        f"{category}: {sampled.shape[1]} → {pca.n_components_} dimensions "
        f"({pca.explained_variance_ratio_.sum():.4f} variance)"
    )
    del sampled
    gc.collect()
    return pca


PCAS = {
    category: fit_category_pca(category, AUGMENTATION_ENABLED)
    for category in CATEGORIES
}
pca_log = pd.DataFrame([
    {
        "category": category, "original_dim": pca.n_features_in_,
        "reduced_dim": pca.n_components_,
        "explained_variance": pca.explained_variance_ratio_.sum(),
    }
    for category, pca in PCAS.items()
])
display(pca_log)
pca_log.to_csv(RESULT_ROOT / "pca_dimensions.csv", index=False)


## Phase 4 — PatchCore coreset and KNN baseline


In [ ]:
class L2Index:
    def __init__(self, memory, k=1, prefer_gpu=True):
        self.memory = np.ascontiguousarray(memory.astype(np.float32))
        self.k = min(k, len(self.memory))
        self.backend = "sklearn"
        if faiss is not None:
            cpu_index = faiss.IndexFlatL2(self.memory.shape[1])
            cpu_index.add(self.memory)
            self.index = cpu_index
            self.backend = "faiss-cpu"
            if prefer_gpu and torch.cuda.is_available() and hasattr(faiss, "StandardGpuResources"):
                try:
                    self.resources = faiss.StandardGpuResources()
                    self.index = faiss.index_cpu_to_gpu(self.resources, 0, cpu_index)
                    self.backend = "faiss-gpu"
                except Exception as error:
                    print("FAISS GPU unavailable; using CPU:", error)
        else:
            self.index = NearestNeighbors(n_neighbors=self.k, metric="euclidean")
            self.index.fit(self.memory)

    def search(self, query):
        query = np.ascontiguousarray(query.astype(np.float32))
        if self.backend.startswith("faiss"):
            squared, _ = self.index.search(query, self.k)
            return np.sqrt(np.maximum(squared, 0.0))
        distances, _ = self.index.kneighbors(query, n_neighbors=self.k)
        return distances


def approximate_kcenter_greedy(features, n_select, projection_dim=64, pool_max=50_000):
    # Farthest-first traversal after a fixed random projection. For T4
    # feasibility it traverses a reproducible candidate pool, then returns
    # original (not projected) PCA features.
    rng = np.random.default_rng(SEED)
    pool_size = min(len(features), max(n_select * 10, min(pool_max, len(features))))
    pool_indices = rng.choice(len(features), pool_size, replace=False)
    candidates = torch.from_numpy(features[pool_indices]).float().to(DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    projection = torch.randn(
        candidates.shape[1], min(projection_dim, candidates.shape[1]),
        generator=generator, device=DEVICE,
    ) / math.sqrt(min(projection_dim, candidates.shape[1]))
    projected = candidates @ projection
    center = projected.mean(0, keepdim=True)
    min_distance = ((projected - center) ** 2).sum(1)
    selected = []
    for _ in tqdm(range(min(n_select, pool_size)), desc="k-center coreset"):
        index = int(torch.argmax(min_distance))
        selected.append(index)
        distance = ((projected - projected[index]) ** 2).sum(1)
        min_distance = torch.minimum(min_distance, distance)
        min_distance[index] = -1
    return features[pool_indices[np.asarray(selected)]]


def reduce_training_by_image(category, pca, augment):
    records = []
    for features, paths in iter_cached(category, "train", augment):
        for feature_grid, path in zip(features, paths):
            flat = feature_grid.reshape(-1, feature_grid.shape[-1])
            records.append((path, pca.transform(flat).astype(np.float32)))
    return records


def build_category_models(category, pca, augment=False, rebuild=False):
    category_dir = ARTIFACT_ROOT / category
    patch_path = category_dir / "patchcore_memory.npy"
    knn_path = category_dir / "knn_memory.npy"
    config_path = category_dir / "config.json"
    if all(path.exists() for path in (patch_path, knn_path, config_path)) and not rebuild:
        config = json.loads(config_path.read_text())
        patch_memory, knn_memory = np.load(patch_path), np.load(knn_path)
    else:
        records = reduce_training_by_image(category, pca, augment)
        original_paths = sorted(path for path, _ in records if "::aug" not in path)
        rng = np.random.default_rng(SEED)
        calibration_count = max(1, round(len(original_paths) * CALIBRATION_FRACTION))
        calibration_paths = set(rng.choice(original_paths, calibration_count, replace=False))
        bank = np.concatenate([
            patches for path, patches in records
            if path.split("::aug")[0] not in calibration_paths
        ])
        calibration = [
            patches for path, patches in records if path in calibration_paths
        ]
        coreset_size = min(CORESET_MAX, max(1, round(len(bank) * CORESET_RATIO)))
        patch_memory = approximate_kcenter_greedy(bank, coreset_size)
        knn_count = min(KNN_MEMORY_MAX, len(bank))
        knn_memory = bank[rng.choice(len(bank), knn_count, replace=False)]
        patch_index = L2Index(patch_memory, k=1)
        knn_index = L2Index(knn_memory, k=5)
        patch_calibration = [
            patch_index.search(item)[:, 0].max() for item in calibration
        ]
        knn_calibration = [
            knn_index.search(item).mean(1).max() for item in calibration
        ]
        thresholds = {
            "patchcore": float(np.quantile(patch_calibration, CALIBRATION_QUANTILE)),
            "knn": float(np.quantile(knn_calibration, CALIBRATION_QUANTILE)),
        }
        np.save(patch_path, patch_memory.astype(np.float32))
        np.save(knn_path, knn_memory.astype(np.float32))
        config = {
            "category": category, "img_size": IMG_SIZE,
            "patch_stride": PATCH_STRIDE, "gaussian_sigma": GAUSSIAN_SIGMA,
            "pca_components": int(pca.n_components_),
            "coreset_ratio": CORESET_RATIO,
            "patchcore_memory_size": len(patch_memory),
            "knn_memory_size": len(knn_memory),
            "calibration_normal_images": len(calibration),
            "threshold_policy": f"{CALIBRATION_QUANTILE:.0%} quantile of held-out normal image scores",
            "thresholds": thresholds,
        }
        config_path.write_text(json.dumps(config, indent=2))
    return {
        "patchcore": L2Index(patch_memory, k=1),
        "knn": L2Index(knn_memory, k=5),
        "config": config,
    }


MODELS = {
    category: build_category_models(category, PCAS[category], AUGMENTATION_ENABLED)
    for category in CATEGORIES
}
display(pd.DataFrame([bundle["config"] for bundle in MODELS.values()]))


## Phase 5 — Pixel heatmaps and overlays


In [ ]:
def anomaly_map_from_scores(scores, grid_shape):
    anomaly_map = cv2.resize(
        scores.reshape(grid_shape), (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_CUBIC,
    )
    return cv2.GaussianBlur(
        anomaly_map.astype(np.float32), (0, 0),
        sigmaX=GAUSSIAN_SIGMA, sigmaY=GAUSSIAN_SIGMA,
    )


def overlay_heatmap(image_path, anomaly_map):
    image = np.asarray(
        Image.open(image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    )
    normalized = anomaly_map - anomaly_map.min()
    normalized /= normalized.max() + 1e-8
    heat = cv2.applyColorMap(np.uint8(normalized * 255), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image, 0.58, heat, 0.42, 0)


def resolve_mask_path(image_path):
    image_path = Path(image_path)
    if image_path.parent.name == "good":
        return None
    mask_dir = DATA_ROOT / image_path.parents[2].name / "ground_truth" / image_path.parent.name
    candidates = [mask_dir / f"{image_path.stem}_mask.png", mask_dir / image_path.name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = list(mask_dir.glob(f"{image_path.stem}*.png"))
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"No unique mask for {image_path}")


## Phase 6 — Configurable classic augmentation


In [ ]:
# Keep this False for the first correctness pass. For the final full run,
# set True and execute this cell. It creates 3 on-the-fly augmented copies
# per nominal image (4x total) and rebuilds only training-dependent caches.
# This is a practical robustness/storage tradeoff for a Colab T4.
RUN_FINAL_AUGMENTED = False

if RUN_FINAL_AUGMENTED:
    AUGMENTATION_ENABLED = True
    for category in CATEGORIES:
        extract_and_cache(category, "train", augment=True, rebuild=True)
        PCAS[category] = fit_category_pca(category, augment=True, rebuild=True)
        MODELS[category] = build_category_models(
            category, PCAS[category], augment=True, rebuild=True
        )
    print("Augmented feature caches, PCAs, and memory banks rebuilt.")
else:
    print("Initial pipeline mode: augmentation remains OFF.")


## Phase 7 — Evaluation, tables, plots, heatmap examples, and T4 latency


In [ ]:
def compute_aupro(gt_masks, anomaly_maps, max_fpr=0.30, steps=200):
    gt = np.asarray(gt_masks, dtype=bool)
    scores = np.asarray(anomaly_maps, dtype=np.float32)
    regions = []
    for image_index, mask in enumerate(gt):
        count, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
        for region_id in range(1, count):
            regions.append((image_index, labels == region_id))
    if not regions:
        return float("nan")
    normal = ~gt
    thresholds = np.quantile(scores, np.linspace(1.0, 0.0, steps))
    curve = []
    for threshold in thresholds:
        prediction = scores >= threshold
        fpr = prediction[normal].mean()
        if fpr <= max_fpr:
            pro = np.mean([
                prediction[image_index][region].mean()
                for image_index, region in regions
            ])
            curve.append((float(fpr), float(pro)))
    if not curve:
        return 0.0
    curve = sorted(curve)
    x = np.asarray([0.0] + [point[0] for point in curve] + [max_fpr])
    y = np.asarray([0.0] + [point[1] for point in curve] + [curve[-1][1]])
    x = np.clip(x, 0, max_fpr)
    return float(np.trapz(y, x) / max_fpr)


def metric_row(category, method, labels, image_scores, masks, maps, threshold, latency):
    labels = np.asarray(labels, dtype=int)
    image_scores = np.asarray(image_scores)
    masks = np.asarray(masks, dtype=bool)
    maps = np.asarray(maps, dtype=np.float32)
    predictions = image_scores >= threshold
    positives = labels == 1
    fnr = float(((~predictions) & positives).sum() / max(positives.sum(), 1))
    return {
        "category": category,
        "method": "PatchCore" if method == "patchcore" else "KNN baseline",
        "image_auroc": roc_auc_score(labels, image_scores),
        "pixel_auroc": roc_auc_score(masks.ravel(), maps.ravel()),
        "aupro_0.30": compute_aupro(masks, maps),
        "false_negative_rate": fnr,
        "threshold": threshold,
        "inference_ms": latency,
    }


@torch.inference_mode()
def infer_path(image_path, category, method="patchcore"):
    started = time.perf_counter()
    image = Image.open(image_path).convert("RGB")
    batch = BASE_TRANSFORM(image).unsqueeze(0).to(DEVICE)
    feature_grid = EXTRACTOR(batch)[0].cpu().numpy()
    h, w, channels = feature_grid.shape
    reduced = PCAS[category].transform(feature_grid.reshape(-1, channels))
    distances = MODELS[category][method].search(reduced)
    patch_scores = distances[:, 0] if method == "patchcore" else distances.mean(1)
    anomaly_map = anomaly_map_from_scores(patch_scores, (h, w))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - started) * 1000
    return float(patch_scores.max()), anomaly_map, elapsed_ms


def benchmark_latency(category, method, paths, warmup=3, repeats=20):
    selected = list(paths)[: max(warmup + repeats, 1)]
    for path in selected[:warmup]:
        infer_path(path, category, method)
    timings = [
        infer_path(path, category, method)[2]
        for path in selected[warmup:warmup + repeats]
    ]
    return float(np.mean(timings)) if timings else float("nan")


def evaluate_category(category):
    pca, bundle = PCAS[category], MODELS[category]
    labels, maps = [], {"patchcore": [], "knn": []}
    image_scores = {"patchcore": [], "knn": []}
    masks, paths_seen = [], []
    for features, paths in iter_cached(category, "test", augment=False):
        for feature_grid, path in zip(features, paths):
            h, w, channels = feature_grid.shape
            reduced = pca.transform(feature_grid.reshape(-1, channels))
            is_bad = Path(path).parent.name != "good"
            labels.append(int(is_bad))
            paths_seen.append(path)
            mask_path = resolve_mask_path(path)
            if mask_path:
                mask = np.asarray(
                    Image.open(mask_path).convert("L").resize(
                        (IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST
                    )
                ) > 0
            else:
                mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
            masks.append(mask)
            for method in ("patchcore", "knn"):
                distances = bundle[method].search(reduced)
                patch_scores = (
                    distances[:, 0] if method == "patchcore" else distances.mean(1)
                )
                image_scores[method].append(float(patch_scores.max()))
                maps[method].append(anomaly_map_from_scores(patch_scores, (h, w)))

    rows = []
    for method in ("patchcore", "knn"):
        latency = benchmark_latency(category, method, paths_seen)
        rows.append(metric_row(
            category, method, labels, image_scores[method], masks, maps[method],
            bundle["config"]["thresholds"][method], latency,
        ))

    example_dir = RESULT_ROOT / "heatmaps" / category
    example_dir.mkdir(parents=True, exist_ok=True)
    for target_label, name in ((0, "good"), (1, "defective")):
        index = labels.index(target_label)
        overlay = overlay_heatmap(paths_seen[index], maps["patchcore"][index])
        Image.fromarray(overlay).save(example_dir / f"{name}_patchcore.png")
    np.savez_compressed(
        RESULT_ROOT / f"{category}_predictions.npz",
        labels=np.asarray(labels), masks=np.asarray(masks, dtype=np.uint8),
        patchcore_scores=np.asarray(image_scores["patchcore"]),
        knn_scores=np.asarray(image_scores["knn"]),
        patchcore_maps=np.asarray(maps["patchcore"], dtype=np.float16),
        knn_maps=np.asarray(maps["knn"], dtype=np.float16),
    )
    return rows


evaluation_rows = []
for category in CATEGORIES:
    print("Evaluating", category)
    evaluation_rows.extend(evaluate_category(category))

per_category = pd.DataFrame(evaluation_rows)
macro = (
    per_category.groupby("method", as_index=False)
    [["image_auroc", "pixel_auroc", "aupro_0.30", "false_negative_rate", "inference_ms"]]
    .mean()
)
macro.insert(0, "category", "Average")
macro["threshold"] = np.nan
summary = pd.concat([per_category, macro], ignore_index=True)
summary.to_csv(RESULT_ROOT / "summary.csv", index=False)
display(summary.style.format({
    "image_auroc": "{:.3f}", "pixel_auroc": "{:.3f}",
    "aupro_0.30": "{:.3f}", "false_negative_rate": "{:.3f}",
    "threshold": "{:.4f}", "inference_ms": "{:.1f}",
}))


In [ ]:
chart_metrics = [
    "image_auroc", "pixel_auroc", "aupro_0.30",
    "false_negative_rate", "inference_ms",
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_data = per_category.copy()
for axis, metric in zip(axes.ravel(), chart_metrics):
    pivot = plot_data.pivot(index="category", columns="method", values=metric)
    pivot.loc[CATEGORIES].plot(kind="bar", ax=axis, rot=0)
    axis.set_title(metric.replace("_", " ").title())
    axis.grid(axis="y", alpha=0.25)
    axis.legend(fontsize=8)
axes.ravel()[-1].axis("off")
fig.suptitle("PatchCore vs KNN — electronics anomaly detection", fontsize=16)
fig.tight_layout()
fig.savefig(RESULT_ROOT / "benchmark_comparison.png", dpi=180, bbox_inches="tight")
plt.show()


## Phase 8 — Validate research artifacts

Training and evaluation are complete when every category has a PCA model,
PatchCore memory bank, configuration file, and a generated summary table.
These files belong to the optional research path; the standalone app does
not require them.


In [ ]:
required_artifacts = []
for category in CATEGORIES:
    required_artifacts.extend([
        ARTIFACT_ROOT / category / "pca.joblib",
        ARTIFACT_ROOT / category / "patchcore_memory.npy",
        ARTIFACT_ROOT / category / "config.json",
    ])
required_artifacts.append(RESULT_ROOT / "summary.csv")

missing = [str(path) for path in required_artifacts if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Complete Phases 2-7 before validating the research run. Missing:\n"
        + "\n".join(missing)
    )

manifest = {
    "categories": CATEGORIES,
    "files": [
        {
            "path": str(path.relative_to(PROJECT_ROOT)),
            "bytes": path.stat().st_size,
        }
        for path in required_artifacts
    ],
}
(ARTIFACT_ROOT / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("Research outputs validated and manifest written.")


### Run the standalone inspector

The runnable app uses reference registration and robust change detection;
it does not wait for this notebook. On Windows, double-click
`Start PCB Inspector.bat`. From a terminal, run:

```bash
python -m pip install -r requirements.txt
python -m streamlit run streamlit_app.py
```


## Repository outputs

- `data/` — local VisA PCB images; intentionally ignored by Git.
- `.cache/` — extracted feature grids; intentionally ignored by Git.
- `artifacts/` — optional PCA and memory-bank research outputs.
- `results/` — generated research tables, plots, and heatmaps.
- `streamlit_app.py` — runnable reference-comparison interface.

Run the repository checker before committing:

```bash
python scripts/check_project.py
```
